# ML Pipeline - Production-Ready Fraud Detection

## Genel Bakış

Bu notebook, tüm fraud detection sürecini production'a hazır bir pipeline haline getiriyor. Feature Engineering'de oluşturduğumuz 39 feature ve Model Optimization'da Optuna ile bulduğumuz en iyi XGBoost parametrelerini tek bir sklearn pipeline'ında birleştiriyoruz.

Pipeline yaklaşımının avantajı net: Training sırasında yapılan tüm preprocessing adımları otomatik olarak inference'da da uygulanıyor. Bu, production'da "feature mismatch" veya "preprocessing unutuldu" hatalarını ortadan kaldırıyor.

Bu notebook'ta sklearn-uyumlu transformer'lar oluşturacağız, optimize edilmiş XGBoost'u pipeline'a entegre edeceğiz, cross-validation ile performansı doğrulayacağız ve deployment için modeli serialize edeceğiz.

In [ ]:
import pandas as pd
import numpy as np
import warnings
import os
import json
import joblib

# Sklearn Pipeline Components
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel, VarianceThreshold
from sklearn.base import BaseEstimator, TransformerMixin

# Model - XGBoost Only (optimized in 05_ModelOptimization)
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, make_scorer
)
from sklearn.model_selection import cross_val_score, StratifiedKFold


warnings.filterwarnings('ignore')
np.random.seed(42)

print("\nLibraries imported successfully")


Libraries imported successfully


## 1. Veri Yükleme

İşlenmiş veriyi ve Model Optimization notebook'undan gelen optimize edilmiş XGBoost parametrelerini yüklüyoruz.

In [2]:
# Load processed data
data_dir = '../../data/processed'
model_dir = '../../models/fraud_detection'

X_train = pd.read_csv(f'{data_dir}/X_train.csv')
X_val = pd.read_csv(f'{data_dir}/X_val.csv')
y_train = pd.read_csv(f'{data_dir}/y_train.csv').squeeze()
y_val = pd.read_csv(f'{data_dir}/y_val.csv').squeeze()

# Load optimized XGBoost parameters from 05_ModelOptimization
with open(f'{model_dir}/optimized/optimization_metadata.json', 'r') as f:
    optimization_meta = json.load(f)
xgb_params = optimization_meta['xgboost']['best_params']

print("DATA LOADED")

print(f"\nX_train shape : {X_train.shape}")
print(f"X_val shape   : {X_val.shape}")
print(f"Fraud rate    : {y_train.mean()*100:.2f}%")
print(f"Features      : {X_train.shape[1]}")

print("\nOPTIMIZED XGBOOST PARAMETERS (from 05_ModelOptimization)")

for param, value in xgb_params.items():
    print(f"  {param}: {value}")

DATA LOADED

X_train shape : (472432, 39)
X_val shape   : (118108, 39)
Fraud rate    : 3.50%
Features      : 39

OPTIMIZED XGBOOST PARAMETERS (from 05_ModelOptimization)
  n_estimators: 873
  learning_rate: 0.08117574064591325
  max_depth: 12
  min_child_weight: 10
  subsample: 0.9645986764257073
  colsample_bytree: 0.7320827996616689
  gamma: 4.7004391847171674e-07
  reg_alpha: 2.3714088929330063e-06
  reg_lambda: 1.2754718345342986
  scale_pos_weight: 27.579586700866283
  random_state: 42
  verbosity: 0
  eval_metric: auc
  tree_method: hist
  n_jobs: -1


---
## 2. Pipeline Oluşturma

XGBoost tree-based bir model olduğu için feature scaling'e ihtiyaç duymuyor. Split'ler feature değerlerine göre yapılıyor, büyüklük önemli değil. Bu yüzden pipeline'ımız sadece classifier adımını içeriyor, gereksiz preprocessing eklememize gerek yok.

In [3]:
# Calculate class weights for imbalanced data
scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

# Pipeline with OPTIMIZED XGBoost (params from 05_ModelOptimization)
# Note: Tree-based models don't require feature scaling

xgb_pipeline = Pipeline([
    ('classifier', XGBClassifier(
        n_estimators=xgb_params.get('n_estimators', 873),
        max_depth=xgb_params.get('max_depth', 12),
        learning_rate=xgb_params.get('learning_rate', 0.0812),
        min_child_weight=xgb_params.get('min_child_weight', 10),
        subsample=xgb_params.get('subsample', 0.9646),
        colsample_bytree=xgb_params.get('colsample_bytree', 0.7321),
        reg_lambda=xgb_params.get('reg_lambda', 1.2755),
        scale_pos_weight=xgb_params.get('scale_pos_weight', scale_pos_weight),
        random_state=42,
        tree_method='hist',
        eval_metric='auc',
        n_jobs=-1,
        verbosity=0
    ))
])

print("\nPIPELINE STRUCTURE")
print("-" * 50)
for name, step in xgb_pipeline.named_steps.items():
    print(f"  {name}: {type(step).__name__}")

Scale pos weight: 27.58

PIPELINE STRUCTURE
--------------------------------------------------
  classifier: XGBClassifier


---
## 3. Pipeline Eğitimi ve Değerlendirme

Pipeline'ı eğitip validation seti üzerinde performansını ölçüyoruz. Train-Val gap'i overfitting kontrolü için izliyoruz.

In [4]:
# Train XGBoost Pipeline
print("Training XGBoost Pipeline...")
xgb_pipeline.fit(X_train, y_train)

# Predictions
y_train_pred_xgb = xgb_pipeline.predict(X_train)
y_val_pred_xgb = xgb_pipeline.predict(X_val)
y_train_proba_xgb = xgb_pipeline.predict_proba(X_train)[:, 1]
y_val_proba_xgb = xgb_pipeline.predict_proba(X_val)[:, 1]

# Metrics
train_auc_xgb = roc_auc_score(y_train, y_train_proba_xgb)
val_auc_xgb = roc_auc_score(y_val, y_val_proba_xgb)

print("\nXGBOOST PIPELINE RESULTS")
print(f"\nTrain AUC : {train_auc_xgb:.4f}")
print(f"Val AUC   : {val_auc_xgb:.4f}")
print(f"Gap       : {train_auc_xgb - val_auc_xgb:.4f}")
print(f"\nVal Precision: {precision_score(y_val, y_val_pred_xgb):.4f}")
print(f"Val Recall   : {recall_score(y_val, y_val_pred_xgb):.4f}")
print(f"Val F1       : {f1_score(y_val, y_val_pred_xgb):.4f}")

Training XGBoost Pipeline...

XGBOOST PIPELINE RESULTS

Train AUC : 1.0000
Val AUC   : 0.9730
Gap       : 0.0270

Val Precision: 0.7683
Val Recall   : 0.7989
Val F1       : 0.7833


---
## 4. Cross-Validation

Tek bir train-val split'e güvenmek riskli. 5-fold stratified CV ile pipeline'ın farklı veri alt kümelerinde tutarlı performans gösterip göstermediğini test ediyoruz. CV skorlarındaki düşük standart sapma, modelin güvenilir olduğunu gösteriyor.

In [5]:
# Combine train and val for full CV
X_full = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_full = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

print("CROSS-VALIDATION (5-Fold StratifiedKFold)")

print(f"\nFull dataset: {len(X_full):,} samples\n")

# CV for XGBoost Pipeline
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_cv_scores = cross_val_score(
    xgb_pipeline, X_full, y_full, 
    cv=cv, scoring='roc_auc', n_jobs=-1
)

print(f"XGBoost Pipeline CV AUC: {xgb_cv_scores.mean():.4f} (+/- {xgb_cv_scores.std():.4f})")
print(f"\nFold scores: {[f'{s:.4f}' for s in xgb_cv_scores]}")

CROSS-VALIDATION (5-Fold StratifiedKFold)

Full dataset: 590,540 samples

XGBoost Pipeline CV AUC: 0.9712 (+/- 0.0009)

Fold scores: ['0.9719', '0.9708', '0.9702', '0.9705', '0.9727']


---
## 5. Feature Selection Pipeline

Production'da daha az feature, daha hızlı inference demek. SelectFromModel ile model-based feature selection yaparak sadece gerçekten işe yarayan feature'ları tutuyoruz. Bu yaklaşım özellikle real-time scoring sistemlerinde kritik, çünkü her milisaniye önemli.

Ek olarak, daha az feature ile model maintenance de kolaylaşıyor. 39 feature'ın hepsini production'da takip etmek yerine, sadece önemli olanları izlemek yeterli oluyor.

In [6]:
# Advanced pipeline with feature selection using optimized XGBoost params

advanced_xgb_pipeline = Pipeline([
    ('feature_selection', SelectFromModel(
        XGBClassifier(
            n_estimators=100,
            max_depth=5,
            random_state=42,
            verbosity=0
        ),
        threshold='median'
    )),
    ('classifier', XGBClassifier(
        n_estimators=xgb_params.get('n_estimators', 873),
        max_depth=xgb_params.get('max_depth', 12),
        learning_rate=xgb_params.get('learning_rate', 0.0812),
        min_child_weight=xgb_params.get('min_child_weight', 10),
        subsample=xgb_params.get('subsample', 0.9646),
        colsample_bytree=xgb_params.get('colsample_bytree', 0.7321),
        reg_lambda=xgb_params.get('reg_lambda', 1.2755),
        scale_pos_weight=xgb_params.get('scale_pos_weight', scale_pos_weight),
        random_state=42,
        tree_method='hist',
        eval_metric='auc',
        n_jobs=-1,
        verbosity=0
    ))
])

print("ADVANCED PIPELINE STRUCTURE")
for name, step in advanced_xgb_pipeline.named_steps.items():
    print(f"  {name}: {type(step).__name__}")

ADVANCED PIPELINE STRUCTURE
  feature_selection: SelectFromModel
  classifier: XGBClassifier


In [7]:
# Train advanced pipeline
print("Training Pipeline with Feature Selection...")
advanced_xgb_pipeline.fit(X_train, y_train)

# Get number of selected features
selector = advanced_xgb_pipeline.named_steps['feature_selection']
n_selected = selector.get_support().sum()
n_original = X_train.shape[1]

# Predictions
y_val_proba_adv = advanced_xgb_pipeline.predict_proba(X_val)[:, 1]
y_val_pred_adv = advanced_xgb_pipeline.predict(X_val)

val_auc_adv = roc_auc_score(y_val, y_val_proba_adv)

print("\nPIPELINE RESULTS")
print("-" * 50)
print(f"Original features  : {n_original}")
print(f"Selected features  : {n_selected} ({n_selected/n_original*100:.1f}%)")
print(f"Val AUC            : {val_auc_adv:.4f}")
print(f"Val Precision      : {precision_score(y_val, y_val_pred_adv):.4f}")
print(f"Val Recall         : {recall_score(y_val, y_val_pred_adv):.4f}")
print(f"Val F1             : {f1_score(y_val, y_val_pred_adv):.4f}")

Training Pipeline with Feature Selection...

PIPELINE RESULTS
--------------------------------------------------
Original features  : 39
Selected features  : 20 (51.3%)
Val AUC            : 0.9735
Val Precision      : 0.7493
Val Recall         : 0.8326
Val F1             : 0.7888


---
## 6. Pipeline Karşılaştırması

İki pipeline'ı karşılaştırıyoruz: tüm feature'ları kullanan ve otomatik feature selection yapan. AUC farkı minimal ise, daha az feature kullanan pipeline tercih edilebilir çünkü daha hafif ve bakımı kolay.

In [8]:
# Comparison table
print("PIPELINE COMPARISON")

print(f"\n{'Pipeline':<30} {'Val AUC':<12} {'Precision':<12} {'Recall':<10} {'F1':<10}")

print(f"{'\nOptimized XGBoost Pipeline':<30} {val_auc_xgb:<12.4f} {precision_score(y_val, y_val_pred_xgb):<12.4f} {recall_score(y_val, y_val_pred_xgb):<10.4f} {f1_score(y_val, y_val_pred_xgb):<10.4f}")
print(f"{'Advanced Pipeline (Selection)':<30} {val_auc_adv:<12.4f} {precision_score(y_val, y_val_pred_adv):<12.4f} {recall_score(y_val, y_val_pred_adv):<10.4f} {f1_score(y_val, y_val_pred_adv):<10.4f}")

# Determine best pipeline
if val_auc_xgb >= val_auc_adv:
    best_pipeline_name = 'Optimized XGBoost Pipeline'
    best_pipeline = xgb_pipeline
    best_auc = val_auc_xgb
else:
    best_pipeline_name = ' Pipeline'
    best_pipeline = advanced_xgb_pipeline
    best_auc = val_auc_adv

print(f"\nBest Pipeline: {best_pipeline_name} (AUC: {best_auc:.4f})")

PIPELINE COMPARISON

Pipeline                       Val AUC      Precision    Recall     F1        

Optimized XGBoost Pipeline    0.9730       0.7683       0.7989     0.7833    
Advanced Pipeline (Selection)  0.9735       0.7493       0.8326     0.7888    

Best Pipeline:  Pipeline (AUC: 0.9735)


---
## 7. Pipeline Kaydetme

Pipeline serialization'ın büyük avantajı şu: Tüm preprocessing ve model adımları tek bir dosyada saklanıyor. Production'da sadece `joblib.load()` ile modeli yükleyip, raw data üzerinde direkt `predict()` çağırabiliyoruz. Ayrı ayrı scaler, imputer, model dosyaları ile uğraşmak yerine tek bir artifact ile işi hallediyoruz. Bu hem deployment'ı basitleştiriyor hem de versiyon yönetimini kolaylaştırıyor.

In [9]:
# Create pipeline directory
pipeline_dir = '../../models/fraud_detection/pipeline'
os.makedirs(pipeline_dir, exist_ok=True)

# Save pipelines 
joblib.dump(xgb_pipeline, f'{pipeline_dir}/xgb_pipeline.pkl')
joblib.dump(advanced_xgb_pipeline, f'{pipeline_dir}/advanced_xgb_pipeline.pkl')

# Save pipeline metadata
pipeline_metadata = {
    'pipelines': {
        'xgb_pipeline': {
            'val_auc': float(val_auc_xgb),
            'cv_mean_auc': float(xgb_cv_scores.mean()),
            'cv_std_auc': float(xgb_cv_scores.std()),
            'n_features': int(X_train.shape[1]),
            'description': 'Optimized XGBoost pipeline with params from 05_ModelOptimization'
        },
        'advanced_xgb_pipeline': {
            'val_auc': float(val_auc_adv),
            'n_features_original': int(n_original),
            'n_features_selected': int(n_selected),
            'description': 'XGBoost pipeline with automatic feature selection'
        }
    },
    'best_pipeline': 'xgb_pipeline' if val_auc_xgb >= val_auc_adv else 'advanced_xgb_pipeline',
    'feature_columns': X_train.columns.tolist(),
    'optimized_params_source': 'models/fraud_detection/optimized/optimization_metadata.json'
}

with open(f'{pipeline_dir}/pipeline_metadata.json', 'w') as f:
    json.dump(pipeline_metadata, f, indent=2)

print("PIPELINES SAVED")

print(f"\nDirectory: {pipeline_dir}")
print(f"\nFiles:")
for f in os.listdir(pipeline_dir):
    size = os.path.getsize(f'{pipeline_dir}/{f}') / 1024
    print(f"  - {f} ({size:.1f} KB)")

PIPELINES SAVED

Directory: ../../models/fraud_detection/pipeline

Files:
  - advanced_xgb_pipeline.pkl (20923.9 KB)
  - fraud_pipeline.pkl (19469.5 KB)
  - pipeline_metadata.json (1.5 KB)
  - xgb_pipeline.pkl (19300.7 KB)


---
## 8. Pipeline Yükleme Testi

Kaydedilen pipeline'ı yeniden yükleyip aynı sonuçları verdiğini doğruluyoruz. Bu adım kritik çünkü serialization sırasında bir şeyler bozulmuş olabilir. AUC değerlerinin eşleşmesi, pipeline'ın production'a güvenle alınabileceğini gösteriyor.

In [10]:
# Load pipeline and test
loaded_pipeline = joblib.load(f'{pipeline_dir}/xgb_pipeline.pkl')

# Test predictions
y_test_proba = loaded_pipeline.predict_proba(X_val)[:, 1]
loaded_auc = roc_auc_score(y_val, y_test_proba)

print("PIPELINE LOADING TEST")
print("-" * 50)
print(f"Original AUC : {val_auc_xgb:.4f}")
print(f"Loaded AUC   : {loaded_auc:.4f}")
print(f"Match        : {'Yes' if abs(val_auc_xgb - loaded_auc) < 0.0001 else 'No'}")

# Test single prediction
single_sample = X_val.iloc[[0]]
single_pred = loaded_pipeline.predict_proba(single_sample)[0, 1]
print(f"\nSingle sample prediction: {single_pred:.4f}")
print(f"Actual label: {y_val.iloc[0]}")

PIPELINE LOADING TEST
--------------------------------------------------
Original AUC : 0.9730
Loaded AUC   : 0.9730
Match        : Yes

Single sample prediction: 0.0004
Actual label: 0


---
## 9. Production Inference Fonksiyonu

Production'da kullanılacak hazır bir fonksiyon. Sadece DataFrame verip tahmin alıyorsunuz. Risk kategorileri de otomatik atanıyor, böylece operasyon ekibi hangi işlemlere öncelik vermesi gerektiğini hemen görebiliyor.

In [11]:
def predict_fraud(data: pd.DataFrame, pipeline_path: str = None, threshold: float = 0.5):
    """
    Production-ready fraud prediction function.
    
    Parameters:
    -----------
    data : pd.DataFrame
        Input features DataFrame
    pipeline_path : str
        Path to saved pipeline (default: best pipeline)
    threshold : float
        Classification threshold (default: 0.5)
    
    Returns:
    --------
    dict with predictions, probabilities, and risk categories
    """
    if pipeline_path is None:
        pipeline_path = '../../models/fraud_detection/pipeline/xgb_pipeline.pkl'
    
    # Load pipeline
    pipeline = joblib.load(pipeline_path)
    
    # Get predictions
    probabilities = pipeline.predict_proba(data)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    
    # Risk categories
    risk_categories = pd.cut(
        probabilities,
        bins=[0, 0.1, 0.3, 0.5, 0.7, 1.0],
        labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
    )
    
    return {
        'predictions': predictions,
        'probabilities': probabilities,
        'risk_categories': risk_categories
    }


# Test the function
test_results = predict_fraud(X_val.head(10))

print("PRODUCTION INFERENCE TEST")
results_df = pd.DataFrame({
    'Probability': test_results['probabilities'],
    'Prediction': test_results['predictions'],
    'Risk': test_results['risk_categories'],
    'Actual': y_val.head(10).values
})
print(results_df.to_string(index=False))

PRODUCTION INFERENCE TEST
 Probability  Prediction     Risk  Actual
    0.000364           0 Very Low       0
    0.020453           0 Very Low       0
    0.001247           0 Very Low       0
    0.000997           0 Very Low       0
    0.007678           0 Very Low       0
    0.036605           0 Very Low       0
    0.117564           0      Low       0
    0.000017           0 Very Low       0
    0.000125           0 Very Low       0
    0.000446           0 Very Low       0


---
## Özet

İki pipeline oluşturduk ve kaydettik. `xgb_pipeline.pkl` tüm 39 feature'ı kullanıyor, maksimum performans için tercih edilebilir. `advanced_xgb_pipeline.pkl` ise otomatik feature selection yapıyor, daha hafif ve hızlı inference için uygun.

Her iki pipeline da `models/fraud_detection/pipeline/` dizininde, metadata ile birlikte saklanıyor. Production deployment için hazırlar.